# 04 – Prepare Feature Extractors NTHU

Notebook ini menyiapkan 4 backbone (CNN_BASIC, MobileNet, VGG19, Swin) 
sebagai **feature extractor 512-dim per frame** untuk dataset NTHUDDD2.

- Backbone diload dari hasil training di MRL (`models_saved_data_MRL`).
- Arsitektur dimodifikasi ringan agar `forward(return_features=True)` 
  mengembalikan vektor `[batch_size, 512]`.
- Semua parameter backbone di-*freeze* (tidak di-train lagi).
- Di akhir, dilakukan sanity check 1 batch gambar dari `Dataset_nthuddd2_SPLIT`.


# Import & Konfigurasi Dasar

In [1]:
# 1. IMPORT & KONFIGURASI DASAR

import os
import random
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

import timm  # untuk Swin Transformer

print("PyTorch version :", torch.__version__)
print("timm version    :", timm.__version__)

# --- Konfigurasi device (RTX 3060 harus terdeteksi sebagai 'cuda') ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device in use   :", device)

# --- Fungsi seeding (supaya hasil stabil/reproducible) ---
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)


PyTorch version : 2.7.1+cu118
timm version    : 1.0.25
Device in use   : cuda


# Path & Konstanta Global

In [2]:
# 2. PATH & KONSTANTA GLOBAL

# Path utama sesuai struktur yang sudah kita sepakati
BASE_DIR = r"C:\kuliah-sementara\SKRIPSI"

DATASET_MRL_SPLIT = os.path.join(BASE_DIR, "Dataset_MRL_SPLIT")
DATASET_NTHU_SPLIT = os.path.join(BASE_DIR, "Dataset_nthuddd2_eye_SPLIT")
DATASET_NTHU_RAW   = os.path.join(BASE_DIR, "Dataset_nthuddd2")
MODELS_MRL_DIR     = os.path.join(BASE_DIR, "models_saved_data_MRL")

FEATURES_NTHU_DIR  = os.path.join(BASE_DIR, "features_nthuddd2")
os.makedirs(FEATURES_NTHU_DIR, exist_ok=True)  # folder induk fitur (belum ada file besar di sini)

print("MRL Split :", DATASET_MRL_SPLIT)
print("NTHU Split:", DATASET_NTHU_SPLIT)
print("Models    :", MODELS_MRL_DIR)
print("Fitur out :", FEATURES_NTHU_DIR)

# Normalisasi sesuai mean/std MRL yang telah kamu hitung
IMG_MEAN = [0.3772, 0.3772, 0.3772]
IMG_STD  = [0.1544, 0.1544, 0.1544]
IMG_SIZE = 224

# Transform minimal (tanpa augmentasi) untuk sanity check di NTHU
nthu_val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])


MRL Split : C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT
NTHU Split: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_eye_SPLIT
Models    : C:\kuliah-sementara\SKRIPSI\models_saved_data_MRL
Fitur out : C:\kuliah-sementara\SKRIPSI\features_nthuddd2


#  Definisi Arsitektur CNN Basic dengan return_features

In [3]:
# 3. DEFINISI ARSITEKTUR: CNN BASIC DENGAN OPSI return_features

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        # Global Average Pooling -> aman untuk resolusi berapapun
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Catatan penting:
        # Arsitektur classifier DIJAGA sama dengan versi training,
        # supaya .pth lama tetap kompatibel.
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # idx 0
            nn.Linear(128, 512),          # idx 1
            nn.ReLU(),                    # idx 2
            nn.Dropout(0.5),              # idx 3
            nn.Linear(512, num_classes),  # idx 4
        )

    def forward(self, x, return_features: bool = False):
        # Bagian feature extractor (sama persis seperti sebelumnya)
        x = self.features(x)
        x = self.gap(x)        # [B, 128, 1, 1]
        x = self.classifier[0](x)  # Flatten

        # Ambil vektor 512-dim di tengah classifier
        x = self.classifier[1](x)  # Linear(128 -> 512)
        x = self.classifier[2](x)  # ReLU
        x = self.classifier[3](x)  # Dropout
        features_512 = x           # [B, 512]

        logits = self.classifier[4](features_512)  # Linear(512 -> num_classes)

        if return_features:
            return features_512
        return logits


# MobileNet V3 Drowsiness dengan return_features

In [4]:
# 4. DEFINISI ARSITEKTUR: MobileNet V3 DENGAN OPSI return_features

class MobileNetDrowsiness(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = False):
        """
        pretrained=False karena kita akan load bobot khusus MRL dari .pth sendiri.
        """
        super(MobileNetDrowsiness, self).__init__()
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        base_model = models.mobilenet_v3_small(weights=weights)

        # Simpan backbone (features + avgpool) dan classifier yang sudah kamu modifikasi saat training
        self.features = base_model.features
        self.avgpool = base_model.avgpool

        # n_features: output dari GAP bawaan MobileNet
        n_features = base_model.classifier[0].in_features

        # Struktur classifier DIJAGA sama dengan versi training:
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),           # 0
            nn.Linear(n_features, 512),  # 1
            nn.ReLU(),                   # 2
            nn.Dropout(p=0.3),           # 3
            nn.Linear(512, num_classes)  # 4
        )

    def forward(self, x, return_features: bool = False):
        # Meniru forward bawaan MobileNet, tapi kita pegang titik 512-dim
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.classifier[0](x)   # Dropout
        x = self.classifier[1](x)   # Linear -> 512
        x = self.classifier[2](x)   # ReLU
        x = self.classifier[3](x)   # Dropout
        features_512 = x            # [B, 512]

        logits = self.classifier[4](features_512)

        if return_features:
            return features_512
        return logits


# VGG19 Drowsiness dengan return_features

In [5]:
# 5. DEFINISI ARSITEKTUR: VGG19_BN DENGAN OPSI return_features

class VGG19Drowsiness(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = False):
        super(VGG19Drowsiness, self).__init__()
        weights = models.VGG19_BN_Weights.DEFAULT if pretrained else None
        base_model = models.vgg19_bn(weights=weights)

        self.features = base_model.features
        self.avgpool = base_model.avgpool

        # n_features: sebelum classifier bawaan (biasanya 512*7*7 = 25088)
        n_features = base_model.classifier[0].in_features

        # Struktur classifier DIJAGA sama dengan versi training:
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),           # 0
            nn.Linear(n_features, 512),  # 1
            nn.ReLU(),                   # 2
            nn.Dropout(p=0.3),           # 3
            nn.Linear(512, num_classes)  # 4
        )

    def forward(self, x, return_features: bool = False):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.classifier[0](x)   # Dropout
        x = self.classifier[1](x)   # Linear -> 512
        x = self.classifier[2](x)   # ReLU
        x = self.classifier[3](x)   # Dropout
        features_512 = x            # [B, 512]

        logits = self.classifier[4](features_512)

        if return_features:
            return features_512
        return logits


# SwinDrowsiness dengan return_features

In [6]:
# 6. DEFINISI ARSITEKTUR: SWIN TRANSFORMER DENGAN OPSI return_features

class SwinDrowsiness(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = False):
        super(SwinDrowsiness, self).__init__()
        # num_classes=0 -> backbone hanya mengeluarkan fitur
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            num_classes=0
        )
        n_features = self.backbone.num_features

        # Struktur classifier DIJAGA sama dengan versi training:
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),             # 0
            nn.Linear(n_features, 512),  # 1
            nn.ReLU(),                   # 2
            nn.Dropout(0.3),             # 3
            nn.Linear(512, num_classes)  # 4
        )

    def forward(self, x, return_features: bool = False):
        features_backbone = self.backbone(x)  # [B, n_features]

        x = self.classifier[0](features_backbone)  # Dropout
        x = self.classifier[1](x)                  # Linear -> 512
        x = self.classifier[2](x)                  # ReLU
        x = self.classifier[3](x)                  # Dropout
        features_512 = x                           # [B, 512]

        logits = self.classifier[4](features_512)

        if return_features:
            return features_512
        return logits


# Fungsi Helper get_model + Load .pth + Freeze

In [7]:
# 7. FUNGSI PEMANGGIL MODEL + LOAD BOBOT MRL + FREEZE PARAMETER

MODEL_PTH_PATHS = {
    "cnn_basic": os.path.join(MODELS_MRL_DIR, "CNN_BASIC", "CNN_BASIC_BEST.pth"),
    "mobilenet": os.path.join(MODELS_MRL_DIR, "MOBILENET", "MOBILENET_BEST.pth"),
    "vgg19":     os.path.join(MODELS_MRL_DIR, "VGG19", "VGG19_BEST.pth"),
    "swin":      os.path.join(MODELS_MRL_DIR, "SWIN", "SWIN_BEST.pth"),
}

def get_model(model_name: str):
    model_name = model_name.lower()
    if model_name == "cnn_basic":
        model = SimpleCNN(num_classes=2)
    elif model_name == "mobilenet":
        model = MobileNetDrowsiness(num_classes=2)
    elif model_name == "vgg19":
        model = VGG19Drowsiness(num_classes=2)
    elif model_name == "swin":
        model = SwinDrowsiness(num_classes=2)
    else:
        raise ValueError(f"Model {model_name} tidak dikenali.")

    # Load bobot dari training MRL
    pth_path = MODEL_PTH_PATHS[model_name]
    assert os.path.exists(pth_path), f"File .pth tidak ditemukan: {pth_path}"

    state = torch.load(pth_path, map_location="cpu")
    # Jika kamu save pakai dictionary {'model_state_dict': ...}
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    # ====================================================
    # PERBAIKAN: Hapus prefix "model." atau "backbone." jika ada
    # karena struktur arsitektur kita sekarang dibuat lebih rata
    # ====================================================
    new_state = {}
    for k, v in state.items():
        # Kasus 1: Dulu pakai self.model.features..., sekarang langsung self.features...
        if k.startswith("model."):
            new_key = k.replace("model.", "", 1)
        
        # Kasus 2 (opsional tapi aman): Dulu untuk Swin mungkin beda strukturnya
        else:
            new_key = k
            
        new_state[new_key] = v

    # Gunakan strict=False agar kalau ada loss / metrik yang ikut tersimpan, tidak error
    model.load_state_dict(new_state, strict=False)

    # Freeze semua parameter (backbone + classifier)
    for p in model.parameters():
        p.requires_grad = False

    model.to(device)
    model.eval()

    return model

# Test singkat: inisialisasi semua model
cnn_model       = get_model("cnn_basic")
mobilenet_model = get_model("mobilenet")
vgg_model       = get_model("vgg19")
swin_model      = get_model("swin")

print("Semua model berhasil dimuat & difreeze.")


Semua model berhasil dimuat & difreeze.


# Sanity Check: 1 Batch dari NTHU Train

In [8]:
# 8. SANITY CHECK: CEK OUTPUT [batch, 512] DARI NTHU SPLIT

# Dataset NTHU train (tanpa augmentasi, hanya resize+normalize)
nthu_train_dir = os.path.join(DATASET_NTHU_SPLIT, "train")

nthu_train_dataset = datasets.ImageFolder(
    root=nthu_train_dir,
    transform=nthu_val_transform,
)

BATCH_SIZE_TEST = 8  # kecil saja, RTX 3060 kamu sanggup

nthu_train_loader = DataLoader(
    nthu_train_dataset,
    batch_size=BATCH_SIZE_TEST,
    shuffle=True,
    num_workers=4,     # PC-mu kuat; kalau error di Windows, turunkan ke 0/2
    pin_memory=True if device.type == "cuda" else False,
)

images, labels = next(iter(nthu_train_loader))
images = images.to(device)

print("Batch images shape:", images.shape)

with torch.no_grad():
    feats_cnn  = cnn_model(images,       return_features=True)
    feats_mob  = mobilenet_model(images, return_features=True)
    feats_vgg  = vgg_model(images,       return_features=True)
    feats_swin = swin_model(images,      return_features=True)

print("Fitur CNN_BASIC  :", feats_cnn.shape)
print("Fitur MobileNet  :", feats_mob.shape)
print("Fitur VGG19      :", feats_vgg.shape)
print("Fitur Swin       :", feats_swin.shape)


Batch images shape: torch.Size([8, 3, 224, 224])
Fitur CNN_BASIC  : torch.Size([8, 512])
Fitur MobileNet  : torch.Size([8, 512])
Fitur VGG19      : torch.Size([8, 512])
Fitur Swin       : torch.Size([8, 512])


# TEST PENYIMPANAN KE FOLDER

In [9]:
# 9. TEST PENYIMPANAN KE FOLDER (Memastikan path writeable)

test_file_path = os.path.join(FEATURES_NTHU_DIR, "test_write.txt")
try:
    with open(test_file_path, "w") as f:
        f.write("Test write berhasil! Folder features_nthuddd2 siap menerima file.")
    print("BERHASIL! Coba cek folder features_nthuddd2, pasti ada file 'test_write.txt'")
except Exception as e:
    print(f"GAGAL MENYIMPAN: {e}")


BERHASIL! Coba cek folder features_nthuddd2, pasti ada file 'test_write.txt'
